# 06 · RAG opción B: evidencias externas (dev)

> ## EJECUTAR EN KAGGLE
> - **Acelerador:** GPU T4 ×2
> - **Internet:** **activado (imprescindible: consulta Wikipedia)**
> - **Secretos:** `HF_TOKEN` (y `GH_TOKEN` si el repo es privado)
> - **Ejecutar después de:** —
> - **Entradas:** repo clonado; dev; API pública de Wikipedia en español
> - **Tiempo estimado:** recuperación: ~10–15 min en CPU (se puede hacer en local); LLM: ~3 h de GPU
> - **Salidas (descargar y colocar en el repo):** `results/rag/evidencias_dev.jsonl` → `results/rag/`; `results/llm/ragB_dev_*.csv` → `results/llm/`
> - **Si la sesión se corta:** volver a ejecutar; cada experimento reanuda desde su CSV (hay que conservar los CSV parciales, ver `doc/interno/KAGGLE.md`).


Se recuperan evidencias externas para cada noticia y se le dan al modelo antes del texto.

**Fuente:** `WikipediaSource` (API de MediaWiki, sin clave ni coste, con `User-Agent` y pausa entre peticiones), detrás de la interfaz `EvidenceSource` (`src/rag/evidence.py`), que se puede cambiar por otra.

**Reglas:**
- La consulta sale solo del texto (titular o primera frase). `Link` se usa únicamente para **excluir el dominio original** de las evidencias.
- **Fecha:** el corpus no trae la fecha de cada noticia. Se usa una cota por conjunto (`experiments.FECHA_MAX`: 2018-12-31 para dev) y se pide la revisión de Wikipedia anterior a esa fecha, para no leer artículos escritos después. [CONFIRMAR con el equipo.]
- **Sin evidencia** es un caso propio: el prompt lo dice y aclara que no significa «falsa».

**Control:** se ejecuta también el mismo prompt con todas las noticias «sin evidencia» (`NullSource`), para separar el efecto del prompt del efecto de las evidencias.

**Expectativa honesta:** Wikipedia cubre mal noticias locales y bulos concretos; en una prueba con tres titulares de dev devolvió artículos del tema general (o nada). Si no mejora en dev, se documenta como aproximación probada y descartada. **Test no se usa en este notebook.**

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

IN_KAGGLE = Path("/kaggle").exists()
# Rama que contiene el código nuevo (src/ y notebooks 05-06). Cambiar si se sube a otra.
REPO, BRANCH = "https://github.com/Rodrigo-Romay/Practica-TAPLN.git", "feature/RAG"
PAQUETES = ['transformers', 'accelerate', 'bitsandbytes']

if IN_KAGGLE:
    os.environ["HF_HOME"] = "/tmp/hf"   # más disco libre que /kaggle/working
    ROOT = Path("/kaggle/working/Practica-TAPLN")

    def secreto(nombre):
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(nombre)
        except Exception:
            return None

    if not ROOT.exists():
        gh = secreto("GH_TOKEN")   # solo hace falta si el repo es privado
        url = REPO.replace("https://", f"https://{gh}@") if gh else REPO
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, str(ROOT)], check=True)
    if PAQUETES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *PAQUETES], check=True)
    hf = secreto("HF_TOKEN")
    if hf:
        os.environ["HF_TOKEN"] = hf
    else:
        print("Sin HF_TOKEN: solo funcionarán los modelos no restringidos.")
    # Resultados intermedios no versionados (opcional): dataset privado `fakedes-resultados`
    extra = Path("/kaggle/input/fakedes-resultados/results")
    if extra.exists():
        shutil.copytree(extra, ROOT / "results", dirs_exist_ok=True)
else:
    ROOT = Path.cwd().resolve().parent

sys.path.insert(0, str(ROOT / "src"))
if not (ROOT / "data" / "processed" / "train.csv").exists():
    raw = ROOT / "data" / "raw"
    subprocess.run([sys.executable, str(ROOT / "src" / "process_data.py"),
                    "--train", str(raw / "train.xlsx"), "--dev", str(raw / "development.xlsx"),
                    "--test", str(raw / "test.xlsx"), "--out", str(ROOT / "data" / "processed")], check=True)

In [ ]:
import numpy as np
import pandas as pd
import torch

import data_utils as du
import preprocessing as pp
import evaluation as ev
import experiments as ex
import llm
import llm_fewshot as fs

splits = du.load_clean_splits(ROOT)
media_re = pp.media_regex_from_splits(splits)   # nombres de medio de train/dev
HAY_GPU = torch.cuda.is_available()
MAX_TOKENS = 4000   # si un prompt no cabe, se quitan ejemplos (queda en n_ejemplos); la noticia no se recorta
print({k: len(v) for k, v in splits.items()}, "| GPU:", HAY_GPU)

## Recuperación de evidencias (CPU + Internet, reanudable)

In [ ]:
from rag import evidence as evd

RAG = ROOT / "results" / "rag"
ruta_ev = RAG / "evidencias_dev.jsonl"
K_EV = 3
dev_in = ex.inputs_for(splits, media_re, "dev")
evidencias = evd.retrieve_all(dev_in, splits["dev"]["Link"], evd.WikipediaSource(), ruta_ev,
                              k=K_EV, fecha_max=ex.FECHA_MAX["dev"])
t = evd.evidence_table(ruta_ev)
print(f"noticias: {len(t)} | sin evidencia: {(t.n == 0).mean():.1%} | con error de red: {(t.error != '').sum()}")
print("Si hay errores de red: borrar esas líneas del JSONL y volver a ejecutar la celda.")
t.n.value_counts().sort_index()

### Revisión manual

**[PENDIENTE]** Mirar 20–30 casos: ¿la evidencia trata del mismo hecho? Anotar la proporción de evidencias relevantes en el Diario; sin esa revisión no se puede interpretar el resultado.

In [ ]:
pd.set_option("display.max_colwidth", 200)
t.sample(min(25, len(t)), random_state=du.SEED)[["Id", "query", "n", "titulos"]]

## Experimentos

In [ ]:
# Referencia a batir: la configuración elegida en 03 para cada modelo (few-shot fijo, k=8)
PROMPT = {"gemma4-31b": "base", "qwen38-27b": "detallado"}
REF = {"gemma4-31b": "ajuste_dev_gemma4-31b__base_k8.csv", "qwen38-27b": "ajuste_dev_qwen38-27b__detallado_k8.csv"}
EXPS_B = [
    ex.experiment(splits, media_re, "dev", modo="rag_b", k=K_EV, evidencias=evidencias, fuente="wikipedia"),
    ex.experiment(splits, media_re, "dev", modo="rag_b", k=K_EV, evidencias={}, fuente="ninguna"),   # control
]
EXPS = {clave: EXPS_B for clave in ex.FINALISTAS}
for e in EXPS_B:
    print(e["experimento"], e["pid"])
con = next((i for i, r in enumerate(dev_in.itertuples(index=False)) if evidencias.get(r.Id)), 0)
print(ex.show_prompt(EXPS_B[0], con))


## Ejecución (GPU)

In [ ]:
MODELOS = list(ex.FINALISTAS)   # gemma4-31b (principal) y qwen38-27b (contraste)
if HAY_GPU:
    for clave in MODELOS:
        ex.run_all(ROOT, EXPS[clave], [clave], borrar_cache=IN_KAGGLE)
else:
    print("Sin GPU: no se ejecutan modelos. Con los CSV ya descargados en results/llm/ se puede ver el resumen.")

## Resultados en dev

In [ ]:
COLS = ["modelo", "decision", "f1_fake", "f1_fake_inf", "f1_fake_sup", "auc", "auc_inf", "auc_sup",
        "precision", "recall", "pct_pred_fake", "s_noticia", "tokens", "completo"]
tabla = ex.dev_table(ROOT, "dev")
if len(tabla):
    tabla = tabla.sort_values("auc", ascending=False)
    tabla.to_csv(ROOT / "results" / "llm" / "ragB_dev_resumen.csv", index=False)
    display(tabla[COLS].round(3).reset_index(drop=True))

In [ ]:
# Comparación pareada (bootstrap) de cada experimento nuevo frente a la configuración elegida en 03
# (few-shot fijo k=8) y frente al zero-shot del piloto, por modelo
filas = []
for clave in MODELOS:
    sistemas = {}
    for nombre, f in [("few-shot k=8 (03)", REF[clave]), ("zero-shot (piloto)", f"piloto_dev_{clave}.csv")]:
        if (ROOT / "results" / "llm" / f).exists():
            sistemas[nombre] = ex.to_predictions(pd.read_csv(ROOT / "results" / "llm" / f), nombre)
    for e in EXPS[clave]:
        p = ex.out_path(ROOT, e, clave)
        if fs.is_complete(p, len(e["inputs"])):
            sistemas[f"{e['experimento']} {e['pid']}"] = ex.to_predictions(pd.read_csv(p), e["pid"])
    nuevos = [s for s in sistemas if s not in ("few-shot k=8 (03)", "zero-shot (piloto)")]
    if nuevos and len(sistemas) > 1:
        c = ev.compare_predictions(ROOT, sistemas, "dev")
        filas.append(c[c["A"].isin(nuevos) ^ c["B"].isin(nuevos)].assign(modelo=clave))
if filas:
    display(pd.concat(filas, ignore_index=True).round(3))


In [ ]:
# ¿Ayuda tener evidencia? Métricas por separado para noticias con y sin evidencia (descriptivo)
for clave in MODELOS:
    p = ex.out_path(ROOT, EXPS_B[0], clave)
    if fs.is_complete(p, len(dev_in)):
        r = pd.read_csv(p)
        for nombre, g in r.groupby(r.n_evidencias > 0):
            print(clave, "con evidencia" if nombre else "sin evidencia",
                  ev.evaluate(g.label, ev.decide_threshold(g.logit_diff), g.logit_diff))

## Lectura de resultados y decisión

**[PENDIENTE: rellenar después de ejecutar.]** Anotar aquí y en `doc/diario_trabajo.md`:

- Si mejora al few-shot fijo k=8 de `03_llm` (y al zero-shot) y si la diferencia es significativa (tabla pareada).
- AUC en las Fake no satíricas (es lo más parecido a test que hay en dev).
- Coste (tokens y s/noticia) y problemas encontrados.
- Si la aproximación se mantiene o se abandona, y por qué. **Test no se usa aquí.**
